# GC-LSTM-GhostNet - Step 7 sampled end-to-end (CPU only)

Validate the attached Parquet dataset, train a bounded two-epoch sample, create real explainability and benchmark artifacts, and verify the complete report contract.


In [ ]:
from pathlib import Path
import base64
import io
import shutil
import subprocess
import sys
import zipfile

PROJECT_DIR = Path("/kaggle/working/Luan-Van-GC-LSTM-GhostNet-CICDDoS2019-v1")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "step7_sampled_end_to_end"
MOUNTED_DATA_CANDIDATES = [
    Path("/kaggle/input/cicddos2019-parquet"),
    Path("/kaggle/input/datasets/dungnguyen28101991/cicddos2019-parquet"),
]
PROJECT_ARCHIVE_B64 = "..."

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ARCHIVE_B64))) as project_zip:
    project_zip.extractall(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

mounted_data_dir = next((path for path in MOUNTED_DATA_CANDIDATES if path.exists()), None)
if mounted_data_dir is None and next(Path("/kaggle/input").rglob("dataset_summary.json"), None):
    # Kaggle may choose a normalized mount slug that differs from the API slug.
    # The data loader recursively selects the validated manifests below this root.
    mounted_data_dir = Path("/kaggle/input")
if mounted_data_dir is not None:
    DATA_DIR = mounted_data_dir
else:
    mounted_entries = sorted(str(path) for path in Path("/kaggle/input").iterdir())
    raise FileNotFoundError(
        "Dataset input is not attached. In the Kaggle editor choose Add Input -> "
        "dungnguyen28101991/cicddos2019-parquet, then Save Version / Run All. "
        f"Current /kaggle/input entries: {mounted_entries}"
    )
print(f"Step 7 sampled end-to-end CPU project ready; using dataset at {DATA_DIR}")


In [ ]:
command = [
    sys.executable, "-m", "src.step6_artifact_smoke",
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--samples-per-file", "2048",
    "--sequence-length", "16",
    "--sequence-stride", "8",
    "--batch-size", "64",
    "--device", "cpu",
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)


In [ ]:
import json

summary_path = OUTPUT_DIR / "step6_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["status"] == "passed", summary
assert summary["device"] == "cpu", summary
assert summary["sequence_leakage_status"] == "passed", summary
assert summary["expected_not_yet_run"] == ["ablation_comparison", "cfaco_convergence"], summary
assert len(summary["report"]["produced"]) == 11, summary
assert (OUTPUT_DIR / "report" / "report_status.json").exists()
assert (OUTPUT_DIR / "artifacts" / "benchmark.json").exists()
step7_acceptance = {
    "status": "passed",
    "step": 7,
    "mode": "sampled_end_to_end",
    "device": "cpu",
    "source_summary": str(summary_path),
    "produced_report_groups": len(summary["report"]["produced"]),
}
(OUTPUT_DIR / "step7_acceptance.json").write_text(
    json.dumps(step7_acceptance, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
step7_acceptance
